In [ ]:
from sklearn.model_selection import train_test_split
import os, glob
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras import Input, Model, Sequential
from tensorflow import keras
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, GlobalAveragePooling2D, Dropout, concatenate, Bidirectional, LSTM
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
from PIL import Image, ImageOps
from sklearn.utils import class_weight
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import os
from pathlib import Path
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

SRC_ROOT = Path(r"E:\fruits")   
DST_ROOT = Path(r"E:\Fruit_Quality_Split")          
MOVE_FILES = False  

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
CLASS_MAP = {"good": 0, "bad": 1, "mixed": 2}

def norm_label(name: str):
    n = name.lower()
    if "good" in n:   return "good"
    if "bad" in n:    return "bad"
    if "mixed" in n:  return "mixed"
    return None

rows = []
for label_dir in SRC_ROOT.iterdir():
    if not label_dir.is_dir():
        continue
    label_key = norm_label(label_dir.name)
    if label_key is None:
        continue

    for p in label_dir.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rows.append({
                "image_id": str(p.resolve()),
                "label": label_key,
                "label_id": CLASS_MAP[label_key],
                "isNight": 1,
            })


df = pd.DataFrame(rows)
assert not df.empty, "Δεν βρέθηκαν εικόνες."
print("Σύνολο:", len(df), " | Ανά κλάση:\n", df["label"].value_counts(), "\n")

train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label_id"], random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=42)

train_df = train_df.assign(split="train")
valid_df = valid_df.assign(split="valid")
test_df  = test_df.assign(split="test")
df_split = pd.concat([train_df, valid_df, test_df], ignore_index=True)

print("Splits:", df_split["split"].value_counts().to_dict())

for split in ["train","valid","test"]:
    for lbl in ["good","bad","mixed"]:
        (DST_ROOT / split / lbl).mkdir(parents=True, exist_ok=True)

new_paths = []
for r in df_split.itertuples(index=False):
    src = Path(r.image_id)
    dst = DST_ROOT / r.split / r.label / src.name

    if MOVE_FILES:
        shutil.move(str(src), str(dst))
    else:
        shutil.copy2(str(src), str(dst))

    new_paths.append(str(dst.resolve()))

df_split = df_split.copy()
df_split["image_id"] = new_paths  

out_csv = DST_ROOT / "annotations_v2_with_label_id.csv"
df_split.to_csv(out_csv, index=False, encoding="utf-8-sig")

print(f"\n✅ Έτοιμο: {out_csv}")
print(pd.crosstab(df_split["split"], df_split["label"]))
print("\nΠαράδειγμα γραμμής:")
print(df_split.sample(1).to_dict(orient="records")[0])

KeyboardInterrupt: 

In [ ]:
DATA_PATH = r'C:/Users/evri/Desktop/Διπλωματικη/Adversarial AI Attack Detection/Adversarial-AI-Attack-Detection'
os.listdir(DATA_PATH)

['.git',
 '.gitignore',
 '1.Merge_Annotations_dayTrain_nightTrain.ipynb',
 '10.Merge_All_Annotations.ipynb',
 '11.Adversarial_Detector_Training_ANN_CNN_RNN.ipynb',
 '12.Evaluation_ANN_CNN_RNN_Clean_Adversarial.ipynb',
 '13.Visualize_Eval_Summaries.ipynb',
 '14.Evaluation_Adversarial_Detector_ANN_CNN_RNN.ipynb',
 '15.Visualize_Eval_Detector_Summaries.ipynb',
 '16.SemiSupervised_Lifelong_CNN_Adversarial_Detector.ipynb',
 '17.Lifelong_Adaptation.ipynb',
 '18.Fruit Quality.ipynb',
 '2.Crop_Images.ipynb',
 '3.ANN.ipynb',
 '4.CNN.ipynb',
 '5.RNN.ipynb',
 '6.Convert_To_YOLO_Format.ipynb',
 '7.YOLOv8.ipynb',
 '8.Adversarial_Attacks_ANN_CNN_RNN.ipynb',
 '9.Build_Adversarial_Annotations.ipynb',
 'adv_annotations',
 'adv_outputs',
 'checkpoints_lifelong',
 'COVID19_Pneumonia_Normal_Chest_Xray_PA_Dataset',
 'evaluation_detector_summary.xlsx',
 'evaluation_detector_unknown_fgsm_RNN_summary.xlsx',
 'evaluation_detector_unknown_PGD_summary.xlsx',
 'evaluation_summary.xlsx',
 'eval_detector_unknown_fg

In [4]:
df = pd.read_csv(os.path.join(DATA_PATH,'fruit_quality_split_annotations.csv'), sep=',')

In [5]:
df.head()

,image_id,label,isNight,split
0,E:\Fruit_Quality_Split\train\bad\IMG_2510.JPG,1,1,train
1,E:\Fruit_Quality_Split\train\good\IMG202007281...,0,1,train
2,E:\Fruit_Quality_Split\train\good\IMG_9493.JPG,0,1,train
3,E:\Fruit_Quality_Split\train\good\20190820_153...,0,1,train
4,E:\Fruit_Quality_Split\train\good\IMG202007281...,0,1,train


In [ ]:
import os, cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from tensorflow.keras.utils import to_categorical

CSV_PATH   = "fruit_quality_split_annotations.csv"
IMG_SIZE   = 128
NUM_CLASSES = 3  

df = pd.read_csv(CSV_PATH)

print("Splits:", df["split"].value_counts().to_dict())
print("Labels per split:\n", pd.crosstab(df["split"], df["label"]))

def build_Xy(df_part):
    X_images, X_isNight, y = [], [], []

    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        img_path = str(row["image_id"]).replace("\\", "/")
        label    = int(row["label_id"] if "label_id" in row else row["label"])
        is_night = int(row["isNight"])   

        if not os.path.exists(img_path):
            print(f"❌ Missing image: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read: {img_path}")
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0
        X_images.append(img)
        X_isNight.append(is_night)
        y.append(label)

    X_images  = np.array(X_images).reshape(len(X_images), -1)
    X_isNight = np.array(X_isNight).reshape(-1, 1)
    X_combined = np.hstack([X_images, X_isNight])  

    y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
    return X_combined, y


X_train, y_train = build_Xy(df[df["split"] == "train"])
X_val,   y_val   = build_Xy(df[df["split"] == "valid"])   
X_test,  y_test  = build_Xy(df[df["split"] == "test"])

print("Shapes:",
      "\n X_train:", X_train.shape, " y_train:", y_train.shape,
      "\n X_val:  ", X_val.shape,   " y_val:  ", y_val.shape,
      "\n X_test: ", X_test.shape,  " y_test: ", y_test.shape)

Splits: {'train': 15620, 'valid': 1953, 'test': 1953}
Labels per split:
 label     0     1    2
split                 
test   1167   679  107
train  9331  5430  859
valid  1166   679  108


100%|██████████| 1953/1953 [00:25<00:00, 77.65it/s] 


Shapes: 
 X_train: (15620, 49153)  y_train: (15620, 3) 
 X_val:   (1953, 49153)  y_val:   (1953, 3) 
 X_test:  (1953, 49153)  y_test:  (1953, 3)


In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),         
    Dense(128, activation='relu'),            
    Dense(64, activation='relu'),              
    Dense(NUM_CLASSES, activation='softmax')    
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │     6,291,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,300,163 (24.03 MB)

 Trainable params: 6,300,163 (24.03 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    shuffle=True
)

Epoch 1/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 12s 47ms/step - accuracy: 0.6518 - loss: 1.5854 - val_accuracy: 0.7194 - val_loss: 0.7525
Epoch 2/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.7223 - loss: 0.6694 - val_accuracy: 0.4255 - val_loss: 2.2827
Epoch 3/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.7114 - loss: 0.7189 - val_accuracy: 0.7373 - val_loss: 0.5965
Epoch 4/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.7416 - loss: 0.5968 - val_accuracy: 0.5781 - val_loss: 0.9220
Epoch 5/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.7648 - loss: 0.5505 - val_accuracy: 0.7573 - val_loss: 0.5822
Epoch 6/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 44ms/step - accuracy: 0.7954 - loss: 0.4934 - val_accuracy: 0.7235 - val_loss: 0.6275
Epoch 7/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.8107 - loss: 0.4580 - val_accuracy: 0.8146 - val_loss: 0.4423
Epoch 8/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 11s 45ms/step - accuracy: 0.8296 - loss: 0.4155 - 

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}")

Test loss: 0.2482 | Test acc: 0.9048


In [ ]:
y_pred = model.predict(X_test, batch_size=512)

y_pred_labels = np.argmax(y_pred, axis=1)
y_true_labels = np.argmax(y_test, axis=1)

print(classification_report(y_true_labels, y_pred_labels, digits=4))

cm = confusion_matrix(y_true_labels, y_pred_labels)
print("Confusion Matrix:\n", cm)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
              precision    recall  f1-score   support

           0     0.9381    0.9220    0.9300      1167
           1     0.8842    0.9219    0.9027       679
           2     0.6633    0.6075    0.6341       107

    accuracy                         0.9048      1953
   macro avg     0.8285    0.8171    0.8223      1953
weighted avg     0.9043    0.9048    0.9043      1953

Confusion Matrix:
 [[1076   65   26]
 [  46  626    7]
 [  25   17   65]]


In [ ]:
model.save("fruit_quality_ann.keras")
print("Model saved to  'fruit_quality_ann.keras'")

Model saved to  'traffic_light_ann_model.keras'


In [ ]:
CSV_PATH   = "fruit_quality_split_annotations.csv"
IMG_SIZE   = 128
NUM_CLASSES = 3 


df = pd.read_csv(CSV_PATH, sep=None, engine="python")

print("Splits:", df["split"].value_counts().to_dict())
print("Labels per split:\n", pd.crosstab(df["split"], df["label"]))

assert df["label"].between(0, NUM_CLASSES-1).all(), "Found labels outside 0..2"

def build_Xy(df_part):
    X_images, X_isNight, y = [], [], []

    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        img_path = str(row["image_id"]).replace("\\", "/")
        label    = int(row["label"])
        is_night = int(row["isNight"])

        if not os.path.exists(img_path):
            print(f"❌ Missing image: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read: {img_path}")
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0

        X_images.append(img)
        X_isNight.append(is_night)
        y.append(label)

    X_images = np.array(X_images).reshape(len(X_images), -1)
    X_isNight = np.array(X_isNight).reshape(-1, 1)
    X_combined = np.hstack([X_images, X_isNight])

    y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
    return X_combined, y

X_train, y_train = build_Xy(df[df["split"] == "train"])
X_val,   y_val   = build_Xy(df[df["split"] == "valid"])
X_test,  y_test  = build_Xy(df[df["split"] == "test"])

print("Shapes:",
      "\n X_train:", X_train.shape, " y_train:", y_train.shape,
      "\n X_val:  ", X_val.shape,   " y_val:  ", y_val.shape,
      "\n X_test: ", X_test.shape,  " y_test: ", y_test.shape)

Splits: {'train': 15620, 'valid': 1953, 'test': 1953}
Labels per split:
 label     0     1    2
split                 
test   1167   679  107
train  9331  5430  859
valid  1166   679  108


100%|██████████| 1953/1953 [00:07<00:00, 253.03it/s]


Shapes: 
 X_train: (15620, 49153)  y_train: (15620, 3) 
 X_val:   (1953, 49153)  y_val:   (1953, 3) 
 X_test:  (1953, 49153)  y_test:  (1953, 3)


In [ ]:
IMG_SIZE = 128
PIXELS = IMG_SIZE * IMG_SIZE * 3  

def split_back(X):
    X_img_flat = X[:, :PIXELS]
    X_img = X_img_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 3).astype("float32")

    X_isn = X[:, -1].reshape(-1, 1).astype("float32")   
    return X_img, X_isn

Ximg_train, isn_train = split_back(X_train)
Ximg_val,   isn_val   = split_back(X_val)
Ximg_test,  isn_test  = split_back(X_test)

print(Ximg_train.shape, isn_train.shape)  

(15620, 128, 128, 3) (15620, 1)


In [ ]:
NUM_CLASSES = y_train.shape[1]

img_in = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = Conv2D(32, 3, activation='relu')(img_in)   
x = MaxPooling2D()(x)                           
x = BatchNormalization()(x)                     

x = Conv2D(64, 3, activation='relu')(x)        
x = MaxPooling2D()(x)                           
x = BatchNormalization()(x)    

x = Conv2D(128, 3, activation='relu')(x)        
x = MaxPooling2D()(x)                              
x = Flatten()(x)                                

isn_in = Input(shape=(1,), name="isNight")      

h = Concatenate()([x, isn_in])                 
h = Dense(128, activation='relu')(h)           
h = Dropout(0.3)(h)                             
out = Dense(NUM_CLASSES, activation='softmax')(h)

cnn_model = Model([img_in, isn_in], out)
cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 128, 128,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 126, 126,  │        896 │ image[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 63, 63,    │          0 │ conv2d_2[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 63, 63,    │        128 │ max_pooling2d[0]… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 61, 61,    │     18,496 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 30, 30,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 30, 30,    │        256 │ max_pooling2d_1[… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 28, 28,    │     73,856 │ batch_normalizat… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 14, 14,    │          0 │ conv2d_4[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 25088)     │          0 │ max_pooling2d_2[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ isNight             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 25089)     │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ isNight[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 128)       │  3,211,520 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 3)         │        387 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,305,539 (12.61 MB)

 Trainable params: 3,305,347 (12.61 MB)

 Non-trainable params: 192 (768.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_cnn = cnn_model.fit(
    [Ximg_train, isn_train], y_train,           
    validation_data=([Ximg_val, isn_val], y_val),
    epochs=50, 
    batch_size=64, 
    shuffle=True,                              
    callbacks=[early_stop], 
    verbose=1
)

Epoch 1/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 79s 315ms/step - accuracy: 0.8318 - loss: 0.6723 - val_accuracy: 0.7819 - val_loss: 0.5044
Epoch 2/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 76s 311ms/step - accuracy: 0.8987 - loss: 0.2469 - val_accuracy: 0.7931 - val_loss: 0.5091
Epoch 3/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 76s 312ms/step - accuracy: 0.9092 - loss: 0.2194 - val_accuracy: 0.9201 - val_loss: 0.2160
Epoch 4/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 75s 306ms/step - accuracy: 0.9146 - loss: 0.2016 - val_accuracy: 0.9237 - val_loss: 0.2357
Epoch 5/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 74s 303ms/step - accuracy: 0.8711 - loss: 0.2822 - val_accuracy: 0.9186 - val_loss: 0.2097
Epoch 6/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 74s 303ms/step - accuracy: 0.9229 - loss: 0.1800 - val_accuracy: 0.9391 - val_loss: 0.1500
Epoch 7/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 75s 305ms/step - accuracy: 0.9313 - loss: 0.1626 - val_accuracy: 0.9360 - val_loss: 0.1485
Epoch 8/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 79s 324ms/step - accuracy: 0.9344 - loss: 0

In [ ]:
test_loss_cnn, test_acc_cnn = cnn_model.evaluate([Ximg_test, isn_test], y_test, verbose=0)
print(f"CNN → Test loss: {test_loss_cnn:.4f} | Test acc: {test_acc_cnn:.4f}")

CNN → Test loss: 0.1409 | Test acc: 0.9539


In [ ]:
y_pred_cnn = cnn_model.predict([Ximg_test, isn_test], batch_size=512, verbose=0)

y_pred_idx = y_pred_cnn.argmax(axis=1)
y_true_idx = y_test.argmax(axis=1)

print("Classification report (CNN):")
print(classification_report(y_true_idx, y_pred_idx, digits=4))

print("Confusion matrix (CNN):")
print(confusion_matrix(y_true_idx, y_pred_idx))

Classification report (CNN):
              precision    recall  f1-score   support

           0     0.9834    0.9674    0.9754      1167
           1     0.9726    0.9426    0.9574       679
           2     0.6395    0.8785    0.7402       107

    accuracy                         0.9539      1953
   macro avg     0.8652    0.9295    0.8910      1953
weighted avg     0.9608    0.9539    0.9562      1953

Confusion matrix (CNN):
[[1129   15   23]
 [   9  640   30]
 [  10    3   94]]


In [24]:
cnn_model.save("fruit_quality_cnn.keras")
 
print("Model saved to 'fruit_quality_cnn.keras'")

Model saved to 'fruit_quality_cnn.keras'


In [ ]:
CSV_PATH   = "fruit_quality_split_annotations.csv"
IMG_SIZE   = 128
NUM_CLASSES = 3  

df = pd.read_csv(CSV_PATH, sep=None, engine="python")

print("Splits:", df["split"].value_counts().to_dict())
print("Labels per split:\n", pd.crosstab(df["split"], df["label"]))

assert df["label"].between(0, NUM_CLASSES-1).all(), "Found labels outside 0..2"

def build_Xy(df_part):
    X_images, X_isNight, y = [], [], []

    for _, row in tqdm(df_part.iterrows(), total=len(df_part)):
        img_path = str(row["image_id"]).replace("\\", "/")
        label    = int(row["label"])
        is_night = int(row["isNight"])

        if not os.path.exists(img_path):
            print(f"❌ Missing image: {img_path}")
            continue

        img = cv2.imread(img_path)
        if img is None:
            print(f"⚠️ Could not read: {img_path}")
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE)).astype("float32") / 255.0

        X_images.append(img)
        X_isNight.append(is_night)
        y.append(label)

    X_images = np.array(X_images).reshape(len(X_images), -1)
    X_isNight = np.array(X_isNight).reshape(-1, 1)
    X_combined = np.hstack([X_images, X_isNight])

    y = to_categorical(np.array(y), num_classes=NUM_CLASSES)
    return X_combined, y

X_train, y_train = build_Xy(df[df["split"] == "train"])
X_val,   y_val   = build_Xy(df[df["split"] == "valid"])
X_test,  y_test  = build_Xy(df[df["split"] == "test"])

print("Shapes:",
      "\n X_train:", X_train.shape, " y_train:", y_train.shape,
      "\n X_val:  ", X_val.shape,   " y_val:  ", y_val.shape,
      "\n X_test: ", X_test.shape,  " y_test: ", y_test.shape)

Splits: {'train': 15620, 'valid': 1953, 'test': 1953}
Labels per split:
 label     0     1    2
split                 
test   1167   679  107
train  9331  5430  859
valid  1166   679  108


100%|██████████| 1953/1953 [00:25<00:00, 77.09it/s] 


Shapes: 
 X_train: (15620, 49153)  y_train: (15620, 3) 
 X_val:   (1953, 49153)  y_val:   (1953, 3) 
 X_test:  (1953, 49153)  y_test:  (1953, 3)


In [ ]:
IMG_SIZE = 128
PIXELS = IMG_SIZE * IMG_SIZE * 3   

def split_back(X):
    X_img_flat = X[:, :PIXELS]
    X_img = X_img_flat.reshape(-1, IMG_SIZE, IMG_SIZE, 3).astype("float32")

    X_isn = X[:, -1].reshape(-1, 1).astype("float32")   
    return X_img, X_isn

Ximg_train, isn_train = split_back(X_train)
Ximg_val,   isn_val   = split_back(X_val)
Ximg_test,  isn_test  = split_back(X_test)

print("Ximg_train:", Ximg_train.shape, "isn_train:", isn_train.shape) 

Ximg_train: (15620, 128, 128, 3) isn_train: (15620, 1)


In [ ]:
def image_to_row_sequence(Ximg):
    N, H, W, C = Ximg.shape
    return Ximg.reshape(N, H, W*C).astype("float32")

Xseq_train = image_to_row_sequence(Ximg_train)
Xseq_val   = image_to_row_sequence(Ximg_val)
Xseq_test  = image_to_row_sequence(Ximg_test)

print("Xseq_train:", Xseq_train.shape)  

Xseq_train: (15620, 128, 384)


In [ ]:
NUM_CLASSES = y_train.shape[1]

seq_in = Input(shape=(IMG_SIZE, IMG_SIZE*3), name="row_sequence")  
isn_in = Input(shape=(1,), name="isNight")

z = Bidirectional(LSTM(64, return_sequences=False))(seq_in)
z = Dropout(0.3)(z) 

h = Concatenate()([z, isn_in])
h = Dense(128, activation='relu')(h)

out = Dense(NUM_CLASSES, activation='softmax')(h)

rnn_model = Model([seq_in, isn_in], out)
rnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
rnn_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ row_sequence        │ (None, 128, 384)  │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 128)       │    229,888 │ row_sequence[0][… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ isNight             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 129)       │          0 │ dropout_1[0][0],  │
│ (Concatenate)       │                   │            │ isNight[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128)       │     16,640 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 3)         │        387 │ dense_8[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 246,915 (964.51 KB)

 Trainable params: 246,915 (964.51 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history_rnn = rnn_model.fit(
    [Xseq_train, isn_train], y_train,           
    validation_data=([Xseq_val, isn_val], y_val),
    epochs=50, 
    batch_size=64, 
    shuffle=True,                               
    callbacks=[early], 
    verbose=1
)


Epoch 1/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 18s 64ms/step - accuracy: 0.7146 - loss: 0.6754 - val_accuracy: 0.7568 - val_loss: 0.5555
Epoch 2/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 15s 63ms/step - accuracy: 0.7656 - loss: 0.5720 - val_accuracy: 0.8126 - val_loss: 0.4863
Epoch 3/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 15s 62ms/step - accuracy: 0.8014 - loss: 0.5093 - val_accuracy: 0.8039 - val_loss: 0.4515
Epoch 4/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 15s 61ms/step - accuracy: 0.8356 - loss: 0.4411 - val_accuracy: 0.8658 - val_loss: 0.3513
Epoch 5/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 64ms/step - accuracy: 0.8638 - loss: 0.3775 - val_accuracy: 0.8571 - val_loss: 0.3575
Epoch 6/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 65ms/step - accuracy: 0.8699 - loss: 0.3572 - val_accuracy: 0.8874 - val_loss: 0.3060
Epoch 7/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 18s 73ms/step - accuracy: 0.8803 - loss: 0.3327 - val_accuracy: 0.8761 - val_loss: 0.3205
Epoch 8/50
245/245 ━━━━━━━━━━━━━━━━━━━━ 16s 66ms/step - accuracy: 0.8921 - loss: 0.2985 - 

In [ ]:
test_loss_rnn, test_acc_rnn = rnn_model.evaluate([Xseq_test, isn_test], y_test, verbose=0)
print(f"RNN → Test loss: {test_loss_rnn:.4f} | Test acc: {test_acc_rnn:.4f}")

RNN → Test loss: 0.1490 | Test acc: 0.9488


In [ ]:
y_pred = rnn_model.predict([Xseq_test, isn_test], batch_size=512, verbose=0)

y_pred_idx = y_pred.argmax(axis=1)
y_true_idx = y_test.argmax(axis=1)

print(classification_report(y_true_idx, y_pred_idx, digits=4))

print(confusion_matrix(y_true_idx, y_pred_idx))

              precision    recall  f1-score   support

           0     0.9577    0.9700    0.9638      1167
           1     0.9642    0.9514    0.9577       679
           2     0.7426    0.7009    0.7212       107

    accuracy                         0.9488      1953
   macro avg     0.8882    0.8741    0.8809      1953
weighted avg     0.9482    0.9488    0.9484      1953

[[1132   18   17]
 [  24  646    9]
 [  26    6   75]]


In [32]:
rnn_model.save("fruit_quality_rnn.keras")

print("Model saved to 'fruit_quality_rnn.keras'")

Model saved to 'fruit_quality_rnn.keras'
